In [15]:
import numpy as np
import pandas as pd

In [61]:
def load_data(path='mnist_784.csv'):

    data = pd.read_csv(path)
    

    data = np.array(data)
    m, n = data.shape
    np.random.shuffle(data)

    data_dev = data[0:1000].T 
    Y_dev = data_dev[-1]
    X_dev = data_dev[0:n-1]
    X_dev = X_dev / 255. 

    data_train = data[1000:m].T
    Y_train = data_train[-1]
    X_train = data_train[0:n-1]
    X_train = X_train / 255. 

    return X_train, Y_train, X_dev, Y_dev

X_train, Y_train, X_test, Y_test = load_data()

In [42]:
print(X_test.shape)

(784, 1000)


In [58]:
def init_params():

    W1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(Z, 0)

def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    A = expZ / np.sum(expZ, axis=0, keepdims=True)
    return A

def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def ReLU_deriv(Z):

    return Z > 0

def one_hot(Y):

    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y.T

def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y):
    m = Y.size
    one_hot_Y = one_hot(Y)
    

    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2)
    
    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1)
    
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2


def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    return np.sum(predictions == Y) / Y.size

def get_loss(A2, Y):
    # m is the number of examples
    m = Y.size
    one_hot_Y = one_hot(Y)
    
    # Categorical Cross Entropy Loss formula:
    # Sum( -y * log(prediction) ) / m
    # We add a tiny value (1e-8) inside log to prevent log(0) errors
    loss = -np.sum(one_hot_Y * np.log(A2 + 1e-8)) / m
    return loss

def gradient_descent(X, Y, alpha, iterations):
    W1, b1, W2, b2 = init_params()
    
    # Lists to store history for plotting later if needed
    loss_history = []
    accuracy_history = []
    
    for i in range(iterations):
        # 1. Forward Prop
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        
        # 2. Backward Prop
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y)
        
        # 3. Update Weights
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        
        # 4. Calculate Metrics
        current_loss = get_loss(A2, Y)
        predictions = get_predictions(A2)
        current_accuracy = get_accuracy(predictions, Y)
        
        # Store them
        loss_history.append(current_loss)
        accuracy_history.append(current_accuracy)
        if i%50==0:
            print(f"Iter: {i:03d} | Loss: {current_loss:.4f} | Acc: {current_accuracy:.4f}")
            
    return W1, b1, W2, b2, loss_history, accuracy_history

In [60]:
W1, b1, W2, b2,loss_history,acc_history = gradient_descent(X_train, Y_train, 0.15, 1000)


def make_predictions(X, W1, b1, W2, b2):
    _, _, _, A2 = forward_prop(W1, b1, W2, b2, X)
    predictions = get_predictions(A2)
    return predictions

dev_predictions = make_predictions(X_test, W1, b1, W2, b2)
print("------------------------")
print(f"Validation Accuracy: {get_accuracy(dev_predictions, Y_test):.4f}")

Iter: 000 | Loss: 3.4899 | Acc: 0.0723
Iter: 050 | Loss: 1.1255 | Acc: 0.6276
Iter: 100 | Loss: 0.7612 | Acc: 0.7567
Iter: 150 | Loss: 0.6348 | Acc: 0.8012
Iter: 200 | Loss: 0.5683 | Acc: 0.8237
Iter: 250 | Loss: 0.5256 | Acc: 0.8386
Iter: 300 | Loss: 0.4952 | Acc: 0.8488
Iter: 350 | Loss: 0.4723 | Acc: 0.8569
Iter: 400 | Loss: 0.4542 | Acc: 0.8629
Iter: 450 | Loss: 0.4394 | Acc: 0.8678
Iter: 500 | Loss: 0.4269 | Acc: 0.8720
Iter: 550 | Loss: 0.4162 | Acc: 0.8755
Iter: 600 | Loss: 0.4068 | Acc: 0.8786
Iter: 650 | Loss: 0.3985 | Acc: 0.8811
Iter: 700 | Loss: 0.3911 | Acc: 0.8835
Iter: 750 | Loss: 0.3843 | Acc: 0.8859
Iter: 800 | Loss: 0.3782 | Acc: 0.8882
Iter: 850 | Loss: 0.3727 | Acc: 0.8901
Iter: 900 | Loss: 0.3676 | Acc: 0.8917
Iter: 950 | Loss: 0.3629 | Acc: 0.8932
------------------------
Validation Accuracy: 0.8890


In [45]:
# DEBUGGING DATA STRUCTURE
path='mnist_784.csv'
data = pd.read_csv(path)

print("Columns:", data.columns[:5]) # See the headers
print("First row:", data.iloc[0, :5]) # See the first few values
print("Last column sample:", data.iloc[0, -1]) # Check the last column

Columns: Index(['pixel1', 'pixel2', 'pixel3', 'pixel4', 'pixel5'], dtype='object')
First row: pixel1    0
pixel2    0
pixel3    0
pixel4    0
pixel5    0
Name: 0, dtype: int64
Last column sample: 5
